# ЛР 1.3 — Приближение табличных функций

## Вариант 13

### Задача I (экспериментальные данные)
- `x = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]`
- `y = [1.0, 1.6, 2.5, 3.6, 5.0, 6.6, 8.5, 10.6, 13.0, 15.6, 18.5]`

Методы аппроксимации:
1. Квадратичная: базис `{1, x, x^2}`
2. Полиномиальная степени 3: базис `{1, x, x^2, x^3}`
3. Дробно-рациональная: `y = a/(x+b) + c`

### Задача II (Паде)
- `f(x) = x/(1+x^2)`, `x in [0,1]`
- Аппроксимация Паде `R_{3,2}(x)`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sympy as sp

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.size"] = 11
np.set_printoptions(precision=6, suppress=True)

## Задача I. Аппроксимация экспериментальных данных

Ниже реализован общий МНК через нормальные уравнения:
\[
(B^T B) c = B^T y,
\]
где `B` — матрица базисных функций.

СКО (RMSE):
\[
\sigma = \sqrt{\frac{1}{N}\sum_{i=1}^N (y_i-\hat y_i)^2}
\]

In [ ]:
x = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=float)
y = np.array([1.0, 1.6, 2.5, 3.6, 5.0, 6.6, 8.5, 10.6, 13.0, 15.6, 18.5], dtype=float)


def rmse(y_true, y_hat):
    return float(np.sqrt(np.mean((y_true - y_hat) ** 2)))


def fit_linear_least_squares(basis_columns, y_values):
    """
    МНК вручную через нормальные уравнения.
    basis_columns: список векторов-столбцов базиса.
    """
    B = np.column_stack(basis_columns)
    A = B.T @ B
    b = B.T @ y_values
    coeffs = np.linalg.solve(A, b)
    return coeffs, B, A, b


def predict_from_basis(coeffs, basis_columns):
    B = np.column_stack(basis_columns)
    return B @ coeffs

### Квадратичная аппроксимация `{1, x, x^2}`

In [ ]:
coef_q, B_q, A_q, b_q = fit_linear_least_squares([np.ones_like(x), x, x**2], y)
y_q = predict_from_basis(coef_q, [np.ones_like(x), x, x**2])
rmse_q = rmse(y, y_q)

print("Коэффициенты [a0, a1, a2] для y = a0 + a1*x + a2*x^2:")
print(coef_q)
print("\nМатрица A и вектор b нормальных уравнений (A c = b):")
print("A =\n", A_q)
print("b =", b_q)
print(f"RMSE (квадратичная): {rmse_q:.6e}")

### Полиномиальная аппроксимация степени 3 `{1, x, x^2, x^3}`

In [ ]:
coef_c3, B_c3, A_c3, b_c3 = fit_linear_least_squares([np.ones_like(x), x, x**2, x**3], y)
y_c3 = predict_from_basis(coef_c3, [np.ones_like(x), x, x**2, x**3])
rmse_c3 = rmse(y, y_c3)

print("Коэффициенты [a0, a1, a2, a3] для y = a0 + a1*x + a2*x^2 + a3*x^3:")
print(coef_c3)
print("\nМатрица A и вектор b нормальных уравнений (A c = b):")
print("A =\n", A_c3)
print("b =", b_c3)
print(f"RMSE (степень 3): {rmse_c3:.6e}")

### Дробно-рациональная аппроксимация `y = a/(x+b) + c`

Модель нелинейна по параметру `b`. Используем полу-линейный подход:
- перебираем `b` на сетке,
- для фиксированного `b` решаем линейную МНК-задачу по `a` и `c`.

Это не `curve_fit`, всё решается вручную через `numpy.linalg.solve`.

In [ ]:
def fit_rational_a_over_x_plus_b_plus_c(x_data, y_data, b_grid):
    best = None
    for b in b_grid:
        if np.any(np.isclose(x_data + b, 0.0)):
            continue

        phi = 1.0 / (x_data + b)
        coeffs, B, A, rhs = fit_linear_least_squares([phi, np.ones_like(x_data)], y_data)
        a, c = coeffs
        y_hat = a / (x_data + b) + c
        e = rmse(y_data, y_hat)

        if best is None or e < best["rmse"]:
            best = {
                "a": float(a),
                "b": float(b),
                "c": float(c),
                "rmse": float(e),
                "A": A,
                "rhs": rhs,
                "y_hat": y_hat,
            }
    return best


b_candidates = np.linspace(0.05, 5.0, 4000)
best_rat = fit_rational_a_over_x_plus_b_plus_c(x, y, b_candidates)

a_rat, b_rat, c_rat = best_rat["a"], best_rat["b"], best_rat["c"]
y_rat = best_rat["y_hat"]
rmse_rat = best_rat["rmse"]

print(f"Лучшие параметры: a={a_rat:.6f}, b={b_rat:.6f}, c={c_rat:.6f}")
print("Матрица A и вектор b (для линейного шага при найденном b):")
print("A =\n", best_rat["A"])
print("b =", best_rat["rhs"])
print(f"RMSE (дробно-рациональная): {rmse_rat:.6e}")

In [ ]:
x_dense = np.linspace(x.min(), x.max(), 500)

# Модели на плотной сетке
y_q_dense = coef_q[0] + coef_q[1] * x_dense + coef_q[2] * x_dense**2
y_c3_dense = coef_c3[0] + coef_c3[1] * x_dense + coef_c3[2] * x_dense**2 + coef_c3[3] * x_dense**3
y_rat_dense = a_rat / (x_dense + b_rat) + c_rat

plt.figure(figsize=(10, 6))
plt.scatter(x, y, c="black", s=35, label="Экспериментальные данные")
plt.plot(x_dense, y_q_dense, "--", lw=2, label="Квадратичная")
plt.plot(x_dense, y_c3_dense, "-.", lw=2, label="Полином степени 3")
plt.plot(x_dense, y_rat_dense, ":", lw=2.5, label="Дробно-рациональная")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Задача I: сравнение аппроксимирующих функций")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

result_task1 = pd.DataFrame(
    {
        "Метод": [
            "Квадратичная {1,x,x^2}",
            "Полином степени 3 {1,x,x^2,x^3}",
            "Дробно-рациональная a/(x+b)+c",
        ],
        "СКО (RMSE)": [rmse_q, rmse_c3, rmse_rat],
    }
).sort_values("СКО (RMSE)")

print("Сравнение точности аппроксимаций (чем меньше СКО, тем лучше):")
display(result_task1)

## Задача II. Аппроксимация Паде `R_{3,2}` для `f(x)=x/(1+x^2)`

Ищем
\[
R_{3,2}(x)=\frac{a_0+a_1x+a_2x^2+a_3x^3}{1+b_1x+b_2x^2},
\]
приравнивая коэффициенты ряда Тейлора до порядка `m+n = 5`.

Ниже система строится автоматически из коэффициентов ряда (через `sympy.series`), а затем решается линейной алгеброй (`numpy.linalg.solve`).

In [ ]:
def pade_from_taylor(func_expr, m, n):
    """
    Строит коэффициенты Паде [m/n] для func_expr вокруг x=0.
    Возвращает (a, b) где:
      P(x)=a0+...+am x^m,
      Q(x)=1+b1 x+...+bn x^n.
    """
    x_sym = sp.symbols("x")
    order = m + n

    series = sp.series(func_expr, x_sym, 0, order + 1).removeO().expand()
    c = [float(series.coeff(x_sym, k)) for k in range(order + 1)]


    unknown_count = (m + 1) + n
    A = np.zeros((order + 1, unknown_count), dtype=float)
    rhs = np.zeros(order + 1, dtype=float)

    for k in range(order + 1):
        if k <= m:
            A[k, k] = -1.0

        for j in range(1, n + 1):
            if k - j >= 0:
                A[k, (m + 1) + (j - 1)] = c[k - j]

       
        rhs[k] = -c[k]

    sol = np.linalg.solve(A, rhs)
    a = sol[: m + 1]
    b = np.concatenate(([1.0], sol[m + 1 :]))

    return a, b, A, rhs, c


def eval_poly(coeffs, x):
    # coeffs: [c0, c1, ..., ck]
    y = np.zeros_like(x, dtype=float)
    for i, ci in enumerate(coeffs):
        y += ci * x**i
    return y


def eval_pade(a, b, x):
    return eval_poly(a, x) / eval_poly(b, x)


x_sym = sp.symbols("x")
f2_expr = x_sym / (1 + x_sym**2) 

m, n = 3, 2
a_pade, b_pade, A_pade, rhs_pade, taylor_coeffs = pade_from_taylor(f2_expr, m, n)

print("Коэффициенты Тейлора c0..c5:")
print(np.array(taylor_coeffs))
print("\nСистема для Паде (A u = rhs):")
print("A =\n", A_pade)
print("rhs =", rhs_pade)
print("\nКоэффициенты P (a0..a3):", a_pade)
print("Коэффициенты Q (1,b1,b2):", b_pade)

x2 = np.linspace(0, 1, 600)
y_true2 = x2 / (1 + x2**2)
y_pade = eval_pade(a_pade, b_pade, x2)

rmse_pade = rmse(y_true2, y_pade)
max_abs_pade = float(np.max(np.abs(y_true2 - y_pade)))

print(f"\nRMSE Паде R_{{3,2}}: {rmse_pade:.6e}")
print(f"Макс. абсолютная ошибка: {max_abs_pade:.6e}")

plt.figure(figsize=(9, 5))
plt.plot(x2, y_true2, "k", lw=2, label="f(x)=x/(1+x^2)")
plt.plot(x2, y_pade, "--", lw=2, label="Паде R_{3,2}(x)")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Задача II: аппроксимация Паде")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(x2, np.abs(y_true2 - y_pade), lw=2)
plt.xlabel("x")
plt.ylabel("|ошибка|")
plt.title("Абсолютная ошибка аппроксимации Паде")
plt.grid(True, alpha=0.3)
plt.show()

## Анализ результата

1. Для **Задачи I** лучший метод определяется минимальным СКО в таблице.
2. По графикам видно, какая модель лучше повторяет тренд данных на всём интервале.
3. Для **Задачи II** аппроксимация Паде `R_{3,2}` хорошо согласуется с исходной функцией на `[0,1]`; точность оценивается через RMSE и максимальную абсолютную ошибку.